In [15]:
! pip3 install tiktoken
! pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [16]:
import importlib
import tiktoken

In [17]:
tokenizer = tiktoken.get_encoding("gpt2")

### CREATING INPUT-TARGET PAIRS

In this section, a data loader is implemented to generate input–target pairs using a sliding window approach.

To begin, the entire The Verdict short story is tokenized using a Byte Pair Encoding (BPE) tokenizer.

In [18]:
with open("the-verdict.txt", "r", encoding = "utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(f"Number of tokens: {len(enc_text)}")

Number of tokens: 5145


Executing the code above returns 5145, which represents the total number of tokens in the dataset after applying the BPE tokenizer.

For demonstration purposes, the first 50 tokens are removed from the dataset to produce a more interesting text sequence for the following steps.

In [ ]:
enc_sample = enc_text[:50]

A simple and intuitive way to create input–target pairs for next-word prediction is to define two variables:

x: input tokens
y: target tokens (input tokens shifted by one position)

The context size determines the number of tokens included in each input sequence.

In [ ]:
context_size = 4 #length of the input
#The context size of 4 means that the model is trained to look at a sequence of 4 words (or tokens)
#to predict the next word in the sequence
#The input x is the first 4 tokens [1,2,3,4], and the target y is the next 4 tokens [2,3,4,5]

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:    {y}")

Input tokens: [40, 367, 2885, 1464]
Target tokens:    [367, 2885, 1464, 1807]


By pairing inputs with their shifted targets, we can construct next-word prediction tasks.

In [ ]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "--->", desired)

[40] -> 367
[40, 367] -> 2885
[40, 367, 2885] -> 1464
[40, 367, 2885, 1464] -> 1807


Everything to the left of the arrow (---->) represents the input sequence provided to the model, while the token ID on the right represents the target the model is expected to predict.

For better interpretability, the previous example is repeated by converting token IDs back into readable text.

In [22]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "->", tokenizer.decode([desired]))

I ->  H
I H -> AD
I HAD ->  always
I HAD always ->  thought


Before converting tokens into embeddings, the next step is to implement an efficient data loader that iterates over the dataset and returns inputs and targets as PyTorch tensors.

The goal is to return:

An input tensor representing the model’s input sequence

 A target tensor representing the expected predictions

### IMPLEMENTING A DATA LOADER

To implement an efficient data pipeline, PyTorch’s built-in Dataset and DataLoader classes are used.

Step 1: Tokenize the entire text

Step 2: Use a sliding window to create overlapping sequences of length max_length

Step 3: Define the dataset length

Step 4: Retrieve individual samples from the dataset

In [23]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

The GPTDatasetV1 class extends PyTorch’s Dataset class and defines how individual samples are retrieved.

Each sample includes:

input_chunk: a sequence of token IDs of length max_length
target_chunk: the corresponding shifted target sequence


The following code uses GPTDatasetV1 to load data in batches using PyTorch’s DataLoader.

Step 1: Initialize the tokenizer

Step 2: Create the dataset

Step 3: Use drop_last=True to discard incomplete batches and maintain training stability

Step 4: Set the number of CPU processes for data loading

In [24]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

To better understand the workflow, the DataLoader is tested with:

Batch size = 1
Context size = 4

This helps build intuition for how the dataset and DataLoader interact.

In [25]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

The DataLoader is converted into a Python iterator to retrieve batches using the built-in next() function.

In [26]:
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

PyTorch version: 2.11.0+cpu
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


Each batch contains two tensors:

Input token IDs
Target token IDs

Since max_length = 4, each tensor contains 4 tokens.

Note: In practice, models are typically trained with much larger context sizes (e.g., 256 or more).

To understand the effect of stride=1, another batch is retrieved from the dataset.

Comparing consecutive batches shows that token sequences shift by one position.

For example:

The second token in the first batch becomes the first token in the next batch

This behavior mimics a sliding window across the dataset.

In [27]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [28]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


Increasing the stride to 4 ensures:

Full dataset utilization (no tokens skipped)
No overlap between batches

Reducing overlap can help minimize overfitting during training.